In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score, cross_val_predict, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Bayesian Models

A Bayesian model is a probabilistic framework for learning and inference that combines prior knowledge with observed evidence to produce updated beliefs, expressed as a posterior distribution. It is grounded in Bayes' theorem.

---
## A review of Bayes' Theorem

### Intuition

Imagine a factory that produces bags of candy in five varieties:
some bags are **100% cherry**, others have a mix, and some are **100% lime**.
You pick a bag at random and start pulling out candies.

You pull out a **lime** candy. How likely is it that your bag is 50% lime? 75% lime?

Your answer depends on two things:
- How common is each bag type at the factory? → **prior**
- How likely is it to pull a lime from *that type* of bag? → **likelihood**

Bayes' theorem formalizes exactly this reasoning:

$$P(\text{bag type} \mid \text{lime}) = \frac{P(\text{lime} \mid \text{bag type}) \cdot P(\text{bag type})}{P(\text{lime})}$$

Each candy you pull is a new piece of evidence that updates your belief about which bag you have.

### General form

$$\boxed{P(h \mid d) = \frac{P(d \mid h) \cdot P(h)}{P(d)}}$$

| Term | Name | Meaning |
|------|------|---------|
| $P(h)$ | **Prior** | Belief about the bag type *before* pulling any candy |
| $P(d \mid h)$ | **Likelihood** | How probable is this candy given that bag type |
| $P(h \mid d)$ | **Posterior** | Updated belief *after* seeing the candy |
| $P(d)$ | **Evidence** | Normalizing constant (same for all hypotheses) |

### Key insight
> Learning = updating beliefs with evidence. Every candy you pull narrows down which bag you have. Today's posterior is tomorrow's prior — each observation builds on the last.

In [ ]:
# Bayes' theorem: updating beliefs with evidence 
# Candy bag example
# Five bag types: h1=0% lime, h2=25%, h3=50%, h4=75%, h5=100% lime

p_lime_given_h = np.array([0.0, 0.25, 0.50, 0.75, 1.0])
prior          = np.array([0.1,  0.2,  0.4,  0.2, 0.1])
labels         = ['h1 (0%)', 'h2 (25%)', 'h3 (50%)', 'h4 (75%)', 'h5 (100%)']
N_obs          = 10

posteriors        = np.zeros((N_obs + 1, 5))
posteriors[0]     = prior.copy()
pred_next         = np.zeros(N_obs + 1)
pred_next[0]      = np.dot(p_lime_given_h, prior)

for n in range(1, N_obs + 1):
    unnorm       = p_lime_given_h * posteriors[n-1]   # likelihood × prior
    norm_factor  = unnorm.sum()                       # P(d) = Σ_h  P(d | h) × P(h) (marginal likelihood) 
    posteriors[n] = unnorm / unnorm.sum()             # normalize → posterior
    pred_next[n]  = np.dot(p_lime_given_h, posteriors[n])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['tab:red','tab:green','tab:blue','tab:orange','tab:purple']
for i, (lbl, col) in enumerate(zip(labels, colors)):
    axes[0].plot(range(N_obs+1), posteriors[:, i], marker='o', markersize=4,
                 label=lbl, color=col)
axes[0].set_xlabel('Lime candies observed'); axes[0].set_ylabel('P(hypothesis | data)')
axes[0].set_title('Posterior probabilities converge to truth')
axes[0].legend(fontsize=8); axes[0].set_ylim([-0.02, 1.02])

axes[1].plot(range(N_obs+1), pred_next, 'o-', color='steelblue')
axes[1].set_xlabel('Lime candies observed')
axes[1].set_ylabel('P(next candy = lime)')
axes[1].set_title('Bayesian prediction improves with data')
axes[1].set_ylim([0.3, 1.05])

plt.suptitle('Bayesian learning — candy bag example (AIMA Fig. 21.1)', y=1.02)
plt.tight_layout()
plt.show()

---
## Maximum a Posteriori (MAP) & Maximum Likelihood

Full Bayesian learning averages predictions over all hypotheses. This gives the Bayes-optimal prediction, but it can be computationally expensive.
**MAP** is a practical shortcut: just pick the **single most probable hypothesis**:

$$h_{MAP} = \underset{h}{\arg\max}\; P(h \mid d) = \underset{h}{\arg\max}\; P(d \mid h) \cdot P(h)$$

When the prior is **uniform** (all hypotheses equally likely), MAP reduces to **Maximum Likelihood (ML)**:

$$h_{ML} = \underset{h}{\arg\max}\; P(d \mid h)$$

### Three approaches compared

| Method | Uses | Pros | Cons |
|--------|------|------|------|
| **Full Bayesian** | All hypotheses weighted | Optimal, handles uncertainty | Computationally expensive |
| **MAP** | Most probable hypothesis | Tractable, uses prior knowledge | Ignores uncertainty in $\theta$ |
| **ML** | Best-fitting hypothesis | Simple, no prior needed | Fails with small data (overfits) |

> MAP naturally penalizes complex hypotheses because there are many of them, each with a small prior.

In [ ]:

# MAP vs Bayesian: when do they differ?
p_lime_given_h = np.array([0.0, 0.25, 0.50, 0.75, 1.0])
prior          = np.array([0.1,  0.2,  0.4,  0.2, 0.1])
h_labels       = ['h1','h2','h3','h4','h5']
N_MAX          = 10

pred_bayes, pred_map, map_hyp = [], [], []
posterior = prior.copy()

for n in range(N_MAX + 1):
    pred_bayes.append(np.dot(p_lime_given_h, posterior))
    idx_map = np.argmax(posterior)
    pred_map.append(p_lime_given_h[idx_map])
    map_hyp.append(h_labels[idx_map])
    if n < N_MAX:
        unnorm    = p_lime_given_h * posterior
        posterior = unnorm / unnorm.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
xs = range(N_MAX + 1)
axes[0].plot(xs, pred_bayes, 'o-', label='Bayesian (all hypotheses)')
axes[0].step(xs, pred_map, where='post', linestyle='--',
             color='tomato', label='MAP (best hypothesis only)')
prev = map_hyp[0]
for i, h in enumerate(map_hyp):
    if h != prev:
        axes[0].axvline(i, color='gray', alpha=0.4, linestyle=':')
        axes[0].text(i+0.1, 0.36, f'MAP→{h}', fontsize=8, color='gray')
        prev = h
axes[0].set_xlabel('Lime candies observed')
axes[0].set_ylabel('P(next candy = lime)')
axes[0].set_title('MAP vs Bayesian predictions')
axes[0].legend(); axes[0].set_ylim([0.3, 1.05])

diff = np.abs(np.array(pred_bayes) - np.array(pred_map))
axes[1].bar(xs, diff, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Observations'); axes[1].set_ylabel('|Bayesian − MAP|')
axes[1].set_title('Gap shrinks as data grow\n(both converge with enough data)')
plt.tight_layout()
plt.show()

print('MAP hypothesis at each step:')
for n, h in enumerate(map_hyp):
    print(f'  n={n:2d} → MAP={h}  pred_MAP={pred_map[n]:.2f}  pred_Bay={pred_bayes[n]:.3f}')

---
# Naive Bayes Classifier

Naive Bayes is a family of probabilistic classifiers known for their speed and simplicity. Despite relying on strong simplifying assumptions, they are surprisingly effective in many real-world scenarios — particularly in high-dimensional settings. They make an excellent starting point before reaching for more complex models.

#### The model

Naive Bayes classifies by applying Bayes' theorem with one strong assumption:
**all features are conditionally independent given the class.**

$$P(C \mid x_1, x_2, \dots, x_n) = \alpha \cdot P(C) \prod_{i=1}^{n} P(x_i \mid C)$$

This lets us estimate each $P(x_i \mid C)$ separately from the data — no joint distributions needed.

#### Why "naive"?
Because the independence assumption is rarely true in practice.
Yet the classifier works surprisingly well despite this — it is **naive but effective**.

#### The zero-probability problem
If a word never appeared in training spam, $P(\text{word} \mid \text{spam}) = 0$ and the entire product becomes 0.
**Laplace smoothing** (add-1) fixes this by adding a small virtual count to every feature:

$$P(x_i \mid C) = \frac{\text{count}(x_i, C) + \alpha}{\text{count}(C) + \alpha \cdot |V|}$$

---
### The three variants

| Variant | Assumes | Use when |
|---------|---------|----------|
| **GaussianNB** | Features follow a Gaussian distribution | Continuous features (measurements, sensor data) |
| **MultinomialNB** | Features are word/event counts | Text classification, document frequency |
| **BernoulliNB** | Features are binary (present/absent) | Short texts, binary feature vectors |


## Gaussian Naive Bayes

This variant is used when the features are **continuous**. It assumes that, within each class, each feature follows a Gaussian distribution:

$$
X_i \mid C \sim \mathcal{N}(\mu_{Ci}, \sigma_{Ci}^2)
$$

Naive Bayes classifies using:

$$
P(C \mid x_1, \dots, x_n) \propto P(C)\prod_{i=1}^n p(x_i \mid C)
$$

where the likelihood of each feature is evaluated with:

$$
p(x_i \mid C) = \mathcal{N}(x_i; \mu_{Ci}, \sigma_{Ci}^2)
$$

During training, the model directly estimates:

- the class priors $P(C)$
- the mean $\mu_{Ci}$ of each feature within each class
- the variance $\sigma_{Ci}^2$ of each feature within each class

During prediction, it computes a score for each class and selects the class with the largest posterior probability.

Let's see how it works with the Iris dataset:

In [ ]:
# GaussianNB — continuous features (Iris dataset)

# Data
iris = load_iris()

X_iris = iris.data[:, [2, 3]]   # petal length, petal width
y_iris = iris.target

# Scatterplot of the full dataset
plt.figure(figsize=(8, 6))

plt.scatter(
    X_iris[:, 0],
    X_iris[:, 1],
    c=y_iris,
    edgecolor="k"
)

plt.xlabel(iris.feature_names[2])
plt.ylabel(iris.feature_names[3])
plt.title("Iris dataset — petal length vs petal width")
plt.show()


In [ ]:

class MyGaussianNB(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape

        self.priors_ = {}
        self.means_ = {}
        self.vars_ = {}

        for c in self.classes_:
            X_c = X[y == c]

            # Prior P(C)
            self.priors_[c] = len(X_c) / n_samples

            # Parameters of p(x_i | C)
            self.means_[c] = X_c.mean(axis=0)
            self.vars_[c] = X_c.var(axis=0) + 1e-9

        return self

    def _log_gaussian_density(self, x, mean, var):
        return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)

    def predict(self, X):
        predictions = []

        for x in X:
            scores = []

            for c in self.classes_:
                # log P(C)
                log_prior = np.log(self.priors_[c])

                # sum_i log p(x_i | C)
                log_likelihood = self._log_gaussian_density(
                    x,
                    self.means_[c],
                    self.vars_[c]
                ).sum()

                # log P(C | x) proportional to:
                # log P(C) + sum_i log p(x_i | C)
                score = log_prior + log_likelihood
                scores.append(score)

            predictions.append(self.classes_[np.argmax(scores)])

        return np.array(predictions)


In [ ]:
my_gnb = MyGaussianNB()

# Cross-validation on the full dataset
# This is the performance estimate for this small dataset.
scores = cross_val_score(
    my_gnb,
    X_iris,
    y_iris,
    cv=5
)

print("CV scores:", scores)
print("Mean CV score:", scores.mean())

# Out-of-fold predictions
# Each prediction is made by a model that did not train on that observation.
y_pred_cv = cross_val_predict(
    my_gnb,
    X_iris,
    y_iris,
    cv=5
)

print("\nCross-validated classification report:")
print(classification_report(y_iris, y_pred_cv))

# Fit one final model on the full dataset only for parameter inspection.
# This model is not used to report performance.
my_gnb.fit(X_iris, y_iris)

print("\nLearned parameters from the final model:")
print("Priors:", my_gnb.priors_)
print("Means:", my_gnb.means_)
print("Variances:", my_gnb.vars_)

In [ ]:
# Define a new flower manually
# Features: [petal length, petal width]
x_new = np.array([[5.0, 1.7]])
prediction = my_gnb.predict(x_new)

print("New flower:", x_new[0])
print("Predicted class:", iris.target_names[prediction[0]])

#### Using sklearn

In [ ]:
gnb = GaussianNB()

gnb_scores = cross_val_score(
    gnb,
    X_iris,
    y_iris,
    cv=5
)

print("CV scores:", gnb_scores)
print("Mean CV score:", gnb_scores.mean())

# Out-of-fold predictions
# Each prediction is made by a model that did not train on that observation.
y_pred_cv = cross_val_predict(
    gnb,
    X_iris,
    y_iris,
    cv=5
)

print("\nCross-validated classification report:")
print(classification_report(y_iris, y_pred_cv))

# Fit the final model on the full dataset only for visualization.
# This model is not used to report performance.
gnb.fit(X_iris, y_iris)

# Create a grid
x_min, x_max = X_iris[:, 0].min() - 0.5, X_iris[:, 0].max() + 0.5
y_min, y_max = X_iris[:, 1].min() - 0.5, X_iris[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

# Predict the grid
Z = gnb.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.25)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, edgecolor="k")

plt.xlabel(iris.feature_names[2])
plt.ylabel(iris.feature_names[3])
plt.title("Decision boundary — GaussianNB")
plt.show()

## Bernoulli Naive Bayes

This variant is used when the features are **binary**. It assumes that each feature indicates presence or absence:

$$
x_i \in \{0,1\}
$$

Naive Bayes classifies using:

$$
P(C \mid x_1, \dots, x_n) \propto P(C)\prod_{i=1}^n P(x_i \mid C)
$$

For each class, the model learns:

$$
\theta_{Ci} = P(x_i = 1 \mid C)
$$

So each feature likelihood is:

$$
P(x_i \mid C) =
\theta_{Ci}^{x_i}(1-\theta_{Ci})^{1-x_i}
$$

During training, the model estimates:

- the class priors $P(C)$
- the probability $\theta_{Ci}$ that each feature appears within each class

During prediction, it computes a score for each class and selects the class with the largest posterior probability.

In SMS spam classification, each feature can represent whether a word appears in the message or not.

Let's see how it works with the SMS spam dataset:

#### Load data

In [ ]:

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

sms = pd.read_csv(
    url,
    sep="\t",
    header=None,
    names=["label", "message"]
)


sms_messages = sms["message"]
sms_labels = sms["label"]


for label in ["ham", "spam"]:
    print(f"\nExamples of {label} messages:")
    examples = sms[sms["label"] == label].sample(5, random_state=42)

    for msg in examples["message"]:
        print("-", msg)

# Count messages per class
class_counts = sms["label"].value_counts()

print(class_counts)

# Bar plot
plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values)

plt.xlabel("Class")
plt.ylabel("Number of messages")
plt.title("Class distribution in SMS dataset")
plt.show()

#### Binary bag of words representation


In [ ]:

vectorizer = CountVectorizer(binary=True)

#  Example data

example_messages = [
    "free prize now",
    "free meeting today"
]

X = vectorizer.fit_transform(example_messages)
print(vectorizer.get_feature_names_out())
print(X.toarray())

In [ ]:
# Real data 
# Train/test split 
X_train_text, X_test_text, y_train, y_test = train_test_split(
    sms_messages,
    sms_labels,
    test_size=0.2,
    random_state=42,
    stratify=sms_labels
)

#### Custom implementation, training and evaluation

In [ ]:

class MyBernoulliNB(BaseEstimator, ClassifierMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing

    def fit(self, X, y):
        # Convert sparse matrix to dense if needed
        if hasattr(X, "toarray"):
            X = X.toarray()

        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape

        self.priors_ = {}
        self.theta_ = {}

        for c in self.classes_:
            X_c = X[y == c]

            # Prior P(C)
            self.priors_[c] = X_c.shape[0] / n_samples

            # Probability that each feature is present:
            # theta_Ci = P(x_i = 1 | C)
            n_docs_c = X_c.shape[0]
            feature_counts = X_c.sum(axis=0)

            self.theta_[c] = (feature_counts + self.alpha) / (n_docs_c + 2 * self.alpha)

        return self

    def predict(self, X):
        if hasattr(X, "toarray"):
            X = X.toarray()

        predictions = []

        for x in X:
            scores = []

            for c in self.classes_:
                # log P(C)
                log_prior = np.log(self.priors_[c])

                # sum_i log P(x_i | C)
                # If x_i = 1 -> log(theta_Ci)
                # If x_i = 0 -> log(1 - theta_Ci)
                log_likelihood = (
                    x * np.log(self.theta_[c]) +
                    (1 - x) * np.log(1 - self.theta_[c])
                ).sum()

                score = log_prior + log_likelihood
                scores.append(score)

            predictions.append(self.classes_[np.argmax(scores)])

        return np.array(predictions)

In [ ]:
bnb_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(binary=True)),
    ("classifier", MyBernoulliNB(alpha=1.0))
])

scores = cross_val_score(
    bnb_pipeline,
    X_train_text,
    y_train,
    cv=5
)

print("CV scores:", scores)
print("Mean CV score:", scores.mean())

In [ ]:
bnb_pipeline.fit(X_train_text, y_train)

y_pred = bnb_pipeline.predict(X_test_text)

print(classification_report(y_test, y_pred))

In [ ]:
# Try some messages

messages = [
    "Congratulations you have won a free prize claim now",
    "URGENT you have won a free prize call now",
    "Hi are we still meeting today"
]

predictions = bnb_pipeline.predict(messages)

for msg, pred in zip(messages, predictions):
    print(msg, "->", pred)

#### Sklearn implementation, training and evaluation

In [ ]:
bnb_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(binary=True)),
    ("classifier", BernoulliNB(alpha=1.0))
])

scores = cross_val_score(
    bnb_pipeline,
    X_train_text,
    y_train,
    cv=5
)

print("CV scores:", scores)
print("Mean CV score:", scores.mean())


bnb_pipeline.fit(X_train_text, y_train)
y_pred = bnb_pipeline.predict(X_test_text)
print(classification_report(y_test, y_pred))

In [ ]:
# Test custom messages
messages = [
    "Congratulations you have won a free prize claim now",
    "URGENT you have won a free prize call now",
    "Hi are we still meeting today"
]

predictions = bnb_pipeline.predict(messages)
probs = bnb_pipeline.predict_proba(messages)

for msg, pred, prob in zip(messages, predictions, probs):
    print("\nMessage:", msg)
    print("Prediction:", pred)
    print("Class probabilities:")
    
    for class_name, p in zip(bnb_pipeline.classes_, prob):
        print(f"  {class_name}: {p:.4f}")

## Multinomial Naive Bayes

This variant is used when the features are **counts**. It assumes that each feature represents how many times something appears:

$$
x_i \in \{0,1,2,\dots\}
$$

Naive Bayes classifies using:

$$
P(C \mid x_1, \dots, x_n) \propto P(C)\prod_{i=1}^n P(x_i \mid C)
$$

For each class, the model learns:

$$
\theta_{Ci} = P(\text{feature } i \mid C)
$$

The class score is computed as:

$$
P(C)\prod_{i=1}^n \theta_{Ci}^{x_i}
$$

During training, the model estimates:

- the class priors $P(C)$
- the probability $\theta_{Ci}$ of each feature within each class

During prediction, it computes a score for each class and selects the class with the largest posterior probability.

In SMS spam classification, each feature can represent how many times a word appears in the message.

Let's see how it works with the SMS spam dataset:

#### Sklearn implementation, training and evaluation

In [ ]:


X_text = sms["message"]
y = sms["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", MultinomialNB(alpha=1.0))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:


# Predictions on test set
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=pipeline.classes_
).plot(ax=axes[0], colorbar=False, cmap="Blues")

axes[0].set_title("Confusion Matrix")

# Top words learned by MultinomialNB
vectorizer = pipeline.named_steps["vectorizer"]
nb = pipeline.named_steps["classifier"]

features = vectorizer.get_feature_names_out()
top = 12

for class_index, class_name in enumerate(nb.classes_):
    # Largest P(word | class)
    idx = nb.feature_log_prob_[class_index].argsort()[-top:][::-1]

    words = features[idx]
    probs = np.exp(nb.feature_log_prob_[class_index][idx])

    y_pos = np.arange(top) + class_index * (top + 1)

    axes[1].barh(y_pos, probs, alpha=0.8, label=class_name)

    for y_i, word, prob in zip(y_pos, words, probs):
        axes[1].text(prob + 0.001, y_i, word, va="center", fontsize=8)

axes[1].set_xlabel("P(word | class)")
axes[1].set_title("Top words learned by MultinomialNB")
axes[1].legend()
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

#### Testing custom messages

In [ ]:
# Test custom messages
messages = [
    "Congratulations you have won a free prize claim now",
    "URGENT you have won a free prize call now",
    "Hi are we still meeting today"
]

predictions = pipeline.predict(messages)
probs = pipeline.predict_proba(messages)

for msg, pred, prob in zip(messages, predictions, probs):
    print("\nMessage:", msg)
    print("Prediction:", pred)
    print("Class probabilities:")

    for class_name, p in zip(pipeline.classes_, prob):
        print(f"  {class_name}: {p:.10f}")

#### Generating random samples



In [ ]:
def generate_message(pipeline, class_label, n_words=12, random_state=None):
    rng = np.random.default_rng(random_state)

    # Get trained vectorizer and Naive Bayes model
    vectorizer = pipeline.named_steps["vectorizer"]
    nb = pipeline.named_steps["classifier"]

    # Vocabulary
    words = vectorizer.get_feature_names_out()

    # P(word | class)
    word_probs = np.exp(nb.feature_log_prob_[list(nb.classes_).index(class_label)])

    # Sample words from the class-specific distribution
    sampled_words = rng.choice(
        words,
        size=n_words,
        replace=True,
        p=word_probs / word_probs.sum()
    )

    return " ".join(sampled_words)

In [ ]:
# Generate random samples (bags) from each class
rng = np.random.randint(0,100)
for label in pipeline.classes_:
    print(f"\nGenerated message from class: {label}")
    print(generate_message(pipeline, label, n_words=12, random_state=rng))


**References:**
- Raschka, Liu & Mirjalili — *Machine Learning with PyTorch and Scikit-Learn* (2022), Ch. 8
- Russell & Norvig — *Artificial Intelligence: A Modern Approach* (2014), Ch. 21.1–21.3
- VanderPlas — Python Data Science Handbook: Essential Tools for Working with Data (2016)